# AI应用基础 · 工单M4.2《工业智能决策助手系统》
**第11周 | 2课时(80分钟) | 理实一体**  | 学生实训版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 实现多轮对话记忆（历史拼接），测试连续追问的上下文保持；示范讲解 |
| **B 组长** | 标准达成 | ≥10问+RAG提升明显+多轮对话+容错通过 |
| **C 操作员** | 基础完成 | 正确回答≥5个典型问题 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）


## 一、课前资讯（Predict·预测）
> **预习快问**：RAG相比纯LLM回答的优势是什么？企业数据合规要注意什么？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4kb"], check=True)
print("数据已生成: data/")

## 三、任务要求（Make）
**实施步骤**
1. 同一问题3版Prompt对比(朴素/角色/示例)
2. KnowledgeBase检索注入CONTEXT再生成
3. Agent: 工具判断→阈值判定→汇总回答
4. 容错: 超时重试+降级话术实测

**验收标准**：及格(D)=`正确回答≥5个典型问题` ；优秀(B+)=`≥10问+RAG提升明显+多轮对话+容错通过`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""M4.2 工业智能决策助手（本地模拟LLM + RAG + Agent）—— 学生代码框架
服务端(教师机): python mock_llm_server.py
本框架默认连 127.0.0.1:8000, 按机房实际IP修改 BASE_URL。
"""
import json
import re

import requests

BASE_URL = "http://127.0.0.1:8000"
KB_PATH = "data/knowledge_base.json"

### Starter 2：`def chat(messages, retries=2, timeout=5):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def chat(messages, retries=2, timeout=5):
    """TODO-1: 调用本地LLM(OpenAI兼容), 带重试与降级(容错必做)"""
    for i in range(retries + 1):
        try:
            r = requests.post(f"{BASE_URL}/v1/chat/completions",
                              json={"messages": messages}, timeout=timeout)
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"]
        except requests.RequestException:
            pass
    return "[降级] 本地服务暂不可用, 已转人工工单通道。"

### Starter 3：`class KnowledgeBase:`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class KnowledgeBase:
    """TODO-2: 知识库检索(关键词或TF-IDF, 二选一并写入决策矩阵)"""

    def __init__(self, path=KB_PATH):
        with open(path, encoding="utf-8") as f:
            self.kb = json.load(f)
        self.docs = [t for items in self.kb.values() for t in items]

    def retrieve(self, query, topk=2):
        """返回最相关的topk条知识(字符串列表)"""
        # === 你的代码 ===
        raise NotImplementedError

### Starter 4：`class Tool:`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class Tool:
    """工具: 名称+功能函数"""

    def __init__(self, name, func):
        self.name, self.func = name, func

    def run(self, *a):
        return self.func(*a)

### Starter 5：`class Agent:`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class Agent:
    """TODO-3: 两步推理: 判断是否查库 → 调用工具组装Prompt → 生成回答"""

    def __init__(self, kb):
        self.kb = kb
        self.tools = {
            "查询知识库": Tool("查询知识库", kb.retrieve),
            "阈值判定": Tool("阈值判定",
                          lambda v: "超标,须停机检查" if float(v) > 4.5
                          else "正常范围"),
        }

    def ask(self, question):
        # 步骤1: 是否含数值判定需求(如"振动7.5mm/s")
        m = re.search(r"([\d.]+)\s*mm/s", question)
        # 步骤2: 组装messages → chat() → 返回回答与依据
        # === 你的代码 ===
        raise NotImplementedError

### Starter 6：`PROMPTS = {`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
PROMPTS = {
    "朴素": "回答问题: {q}",
    "角色设定": "你是资深设备工程师, 按规范回答: {q}",
    "带示例": "示例问: 空压机排气压力低怎么办?\n示例答: 先查气源泄漏…\n现在回答: {q}",
}

### Starter 7：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    kb = KnowledgeBase()
    agent = Agent(kb)
    questions = [
        "泵振动7.5mm/s怎么处理？",
        "轴承温度82℃要不要停机？",
        "变频器过流报警如何排查？",
        "液压油污染度NAS 10级怎么办？",
        "PLC从站通信中断排查顺序？",
    ]
    # ---- 1) Prompt对比(同一问题三版) ----
    for name, tpl in PROMPTS.items():
        msg = [{"role": "user", "content": tpl.format(q=questions[0])}]
        print(f"[{name}] {chat(msg)[:60]}...\n")
    # TODO-4: RAG对比: 检索结果注入后再问, 记录是否更"有依据"
    # TODO-5: Agent全流程问答记录表(问题/回答/依据)
    # for q in questions: print(q, "->", agent.ask(q))

### Starter 8：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M4.2.py` 或本notebook(含输出)
- ⚠ requests必须timeout=5并捕获异常做降级
- ⚠ 企业数据不出域：课内用本地模拟LLM，真实API选做且先脱敏
- ⚠ 检索只取Top2，Prompt过长反而稀释注意力